In [ ]:
# Problema: Definir un contrato para datos tributarios agregados antes de incorporarlos a un análisis por código postal.

from pathlib import Path

import pandas as pd

ROOT = next(path for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (path / 'data').is_dir() and (path / 'submission').is_dir())
SOURCE = ROOT / 'data' / 'vemont.csv'
OUTPUT = ROOT / 'submission' / 'contract_report.csv'


In [ ]:
# La fuente usa una fila por código postal y tramo de ingreso.
tax = pd.read_csv(SOURCE)
tax.shape, tax[['zipcode', 'agi_stub', 'N1', 'A00100']].head()


In [ ]:
# El contrato solo declara lo que el análisis necesita, no las 147 columnas de la fuente.
REQUIRED = {'STATEFIPS', 'STATE', 'zipcode', 'agi_stub', 'N1', 'A00100'}

def validate(frame):
    if not REQUIRED.issubset(frame.columns):
        return 'FAIL', 'BREAKING', 'REJECT'
    if frame['STATE'].ne('VT').any() or frame['STATEFIPS'].ne(50).any():
        return 'FAIL', 'BREAKING', 'REJECT'
    if not frame['agi_stub'].between(1, 6).all() or frame['N1'].lt(0).any():
        return 'FAIL', 'BREAKING', 'REJECT'
    if frame[['zipcode', 'agi_stub']].duplicated().any():
        return 'FAIL', 'BREAKING', 'REJECT'
    change = 'COMPATIBLE' if set(frame.columns) - set(tax.columns) else 'NONE'
    return 'PASS', change, 'ACCEPT'


In [ ]:
# Probamos cambios representativos antes de permitir que un lote entre al pipeline.
cases = {
    'valid': tax,
    'missing_income_group': tax.drop(columns='agi_stub'),
    'wrong_state': tax.assign(STATE='XX'),
    'invalid_income_group': tax.assign(agi_stub=7),
    'optional_column': tax.assign(source_release='2017'),
}
report = pd.DataFrame([(name, *validate(frame)) for name, frame in cases.items()],
                      columns=['batch_name', 'status', 'change_type', 'action'])
report.to_csv(OUTPUT, index=False)
report
